# FU-IDS Experiments — CIC-IDS2017

**Paper:** *FU-IDS: A Federated Unlearning Framework for Distributed Intrusion Detection Systems with Adversarial Unlearning Defense*

**Authors:** M. R. Jan (29200), T. Tahir (29203), A. M. Ghori (29220) — April–May 2026

This notebook runs all five experimental scenarios from the paper end-to-end:
- **Scenario A** — FL Baseline (clean training, FedAvg)
- **Scenario B** — FL with Poisoning (α = 10% label-flip clients)
- **Scenario C** — FU-IDS Recovery (SlideFU calibration unlearning)
- **Scenario D** — Adversarial Unlearning (BadUnlearn vs UnlearnGuard)
- **Scenario E** — Privacy Certification ((ε, δ)-DP noise + MIA verification)

## Quick start
1. Open this in Google Colab.
2. Runtime → Change runtime type → **Hardware accelerator: T4 GPU** (free, much faster).
3. Runtime → **Run all** (Ctrl+F9).
4. The final cell prints a results table you can paste into the paper.

**Compute budget:** ~30 min on CPU, ~5 min on free T4 GPU at default settings.

## Cell 1 — Setup, configuration, seeds

In [ ]:
import os, sys, time, json, math, random, hashlib, copy
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.decomposition import PCA

# ----- reproducibility -----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')

# ----- experiment configuration -----
# Toggle FULL_EXPERIMENT once you have GPU access; default is prototype-scale
FULL_EXPERIMENT = False

@dataclass
class Config:
    # Data
    n_samples: int = 200_000 if FULL_EXPERIMENT else 50_000
    seq_len: int = 1                      # treat each flow as a feature vector (faster than time-series)
    test_frac: float = 0.2
    # FL
    n_clients: int = 20 if FULL_EXPERIMENT else 10
    n_rounds: int = 50 if FULL_EXPERIMENT else 30
    local_epochs: int = 2
    batch_size: int = 256
    lr: float = 1e-3
    # Model
    hidden: int = 128 if FULL_EXPERIMENT else 64
    n_layers: int = 2
    dropout: float = 0.2
    focal_gamma: float = 2.0
    # Adversary
    poison_alpha: float = 0.10            # 10% poisoning clients
    poison_rounds: int = 20
    # SlideFU
    window_W: int = 10
    # UnlearnGuard
    pca_components: int = 8
    trust_tau: float = 0.65
    # DP
    dp_epsilon: float = 1.0
    dp_delta: float = 1e-5

CFG = Config()
print('Configuration:'); [print(f'  {k}: {v}') for k, v in CFG.__dict__.items()]

## Cell 2 — Data: load CIC-IDS2017, or synthesize a CIC-like fallback

**Real-data path (preferred):** Place a single combined CSV named `CICIDS2017.csv` (or any of the day-files from the official UNB release) in your Colab session. The notebook auto-detects it.

**Quickest real-data path on Colab:**
```python
# (run in a separate cell first if you want)
# !pip install kaggle
# from google.colab import files; files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d cicdataset/cicids2017 -p . --unzip
```
**Fallback (no setup needed):** if no CSV is present, this cell synthesizes a 78-feature, 8-class CIC-IDS2017-like dataset with 83% benign / 17% attack ratio. Results from the synthetic set are valid as a proof-of-concept for framework correctness, NOT as final paper numbers — swap in real CIC-IDS2017 before submission.

In [ ]:
def find_cicids_csv() -> str | None:
    candidates = [
        'CICIDS2017.csv',
        'cicids2017.csv',
        'CIC-IDS2017.csv',
        'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
        'data/CICIDS2017.csv'
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # search any csv in cwd that smells like CIC
    for f in os.listdir('.'):
        if f.lower().endswith('.csv') and ('cic' in f.lower() or 'ids' in f.lower()):
            return f
    return None

def synthesize_cicids_like(n: int = CFG.n_samples, n_features: int = 78, seed: int = SEED):
    """Synthesize a CIC-IDS2017-like dataset with realistic structural properties.
    8 classes (1 BENIGN + 7 attack categories), 83% benign / 17% attack imbalance,
    feature distributions vary per class so a model can actually learn them."""
    rng = np.random.RandomState(seed)
    classes = ['BENIGN', 'DoS', 'DDoS', 'BruteForce', 'Botnet', 'Infiltration', 'PortScan', 'WebAttack']
    class_p   = [0.83, 0.05, 0.04, 0.025, 0.02, 0.012, 0.025, 0.018]
    class_p   = np.array(class_p) / sum(class_p)
    y = rng.choice(len(classes), size=n, p=class_p)
    # Feature blocks: each class has a distinct mean vector + shared covariance
    means = rng.uniform(-1.5, 1.5, size=(len(classes), n_features))
    X = np.zeros((n, n_features))
    for c in range(len(classes)):
        idx = (y == c)
        nc = idx.sum()
        X[idx] = rng.normal(loc=means[c], scale=1.0, size=(nc, n_features))
    cols = [f'f{i}' for i in range(n_features)]
    df = pd.DataFrame(X, columns=cols)
    df['Label'] = [classes[i] for i in y]
    return df, classes

csv_path = find_cicids_csv()
if csv_path:
    print(f'Loading real CIC-IDS2017 from: {csv_path}')
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    if 'Label' not in df.columns:
        # try common alternative
        label_col = [c for c in df.columns if c.lower().endswith('label')][0]
        df = df.rename(columns={label_col: 'Label'})
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    if len(df) > CFG.n_samples:
        df = df.groupby('Label', group_keys=False).apply(lambda g: g.sample(n=min(len(g), max(50, int(CFG.n_samples * len(g) / len(df)))), random_state=SEED))
    classes = sorted(df['Label'].unique().tolist())
    DATA_SOURCE = 'real_cicids2017'
else:
    print('No real CIC-IDS2017 CSV found — generating synthetic CIC-like fallback for end-to-end test.')
    df, classes = synthesize_cicids_like()
    DATA_SOURCE = 'synthetic_cicids_like'

print(f'Dataset: {DATA_SOURCE}')
print(f'Total rows: {len(df):,}')
print(f'Classes ({len(classes)}): {classes}')
print('Class distribution:'); print(df['Label'].value_counts(normalize=True).round(4).to_string())

## Cell 3 — Preprocess and partition into non-IID FL clients

In [ ]:
# Encode labels and features
label_enc = LabelEncoder().fit(df['Label'].astype(str))
y_all = label_enc.transform(df['Label'].astype(str))
feat_cols = [c for c in df.columns if c != 'Label']
X_all = df[feat_cols].select_dtypes(include=[np.number]).values.astype(np.float32)
scaler = StandardScaler().fit(X_all)
X_all = scaler.transform(X_all)
n_features = X_all.shape[1]; n_classes = len(label_enc.classes_)
print(f'X shape: {X_all.shape}  |  n_classes: {n_classes}')

# Test split (held out from FL)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=CFG.test_frac, random_state=SEED, stratify=y_all)
print(f'Train: {X_tr.shape}  Test: {X_te.shape}')

# Non-IID partition: each client gets a biased mixture of classes (Dirichlet-like)
def non_iid_partition(X, y, n_clients: int, alpha: float = 0.5, seed: int = SEED):
    rng = np.random.RandomState(seed)
    n_classes_ = int(y.max() + 1)
    label_idx = [np.where(y == c)[0] for c in range(n_classes_)]
    proportions = rng.dirichlet([alpha] * n_clients, size=n_classes_)  # per class, split across clients
    client_idx = [[] for _ in range(n_clients)]
    for c, idx in enumerate(label_idx):
        rng.shuffle(idx)
        cuts = (np.cumsum(proportions[c]) * len(idx)).astype(int)[:-1]
        splits = np.split(idx, cuts)
        for k, s in enumerate(splits):
            client_idx[k].extend(s.tolist())
    return [np.array(ci) for ci in client_idx]

client_indices = non_iid_partition(X_tr, y_tr, CFG.n_clients, alpha=0.5)
for i, idx in enumerate(client_indices):
    if len(idx) == 0:
        # in rare degenerate Dirichlet draws, give a fallback slice
        client_indices[i] = np.random.choice(len(X_tr), size=max(100, len(X_tr) // (CFG.n_clients * 5)), replace=False)
print(f'Per-client sample counts: {[len(i) for i in client_indices]}')

# Build per-client tensors
def to_tensors(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

client_data = []
for idx in client_indices:
    Xc, yc = to_tensors(X_tr[idx], y_tr[idx])
    client_data.append((Xc, yc))

X_te_t, y_te_t = to_tensors(X_te, y_te)
print('Client data ready.')

## Cell 4 — Local IDS model: BiLSTM + Attention (CPU-friendly)

We use a small BiLSTM + attention head over the feature vector treated as a length-1 sequence.
(Setting `seq_len > 1` would require flow-window construction — we'll do that in the FULL_EXPERIMENT path.)

In [ ]:
class BiLSTMAttnIDS(nn.Module):
    def __init__(self, n_features: int, n_classes: int, hidden: int = CFG.hidden,
                 n_layers: int = CFG.n_layers, dropout: float = CFG.dropout):
        super().__init__()
        # Project flat features into a (seq_len=4) faux-sequence so BiLSTM has something to do
        self.feat_proj = nn.Linear(n_features, hidden * 4)
        self.lstm = nn.LSTM(hidden, hidden, num_layers=n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if n_layers > 1 else 0)
        self.attn = nn.Linear(hidden * 2, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden * 2, n_classes))
    def forward(self, x):
        # x: (B, n_features)
        h = self.feat_proj(x).view(x.size(0), 4, -1)  # (B, 4, hidden)
        out, _ = self.lstm(h)                          # (B, 4, 2*hidden)
        a = torch.softmax(self.attn(out).squeeze(-1), dim=-1)  # (B, 4)
        z = (out * a.unsqueeze(-1)).sum(dim=1)         # (B, 2*hidden)
        return self.head(z)

def focal_loss(logits, target, gamma: float = CFG.focal_gamma, weight=None):
    ce = F.cross_entropy(logits, target, reduction='none', weight=weight)
    p = torch.exp(-ce)
    return ((1 - p) ** gamma * ce).mean()

def evaluate(model, X, y, bs: int = 1024):
    model.eval(); preds = []
    ds = TensorDataset(X, y); loader = DataLoader(ds, batch_size=bs)
    with torch.no_grad():
        for xb, _ in loader:
            preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    p = np.concatenate(preds); yn = y.numpy()
    acc = accuracy_score(yn, p) * 100
    f1  = f1_score(yn, p, average='macro') * 100
    benign_idx = label_enc.transform(['BENIGN'])[0] if 'BENIGN' in list(label_enc.classes_) else 0
    fpr = float(((p != yn) & (yn == benign_idx)).sum()) / max(1, (yn == benign_idx).sum()) * 100
    return acc, f1, fpr

# sanity check
_m = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
print(f'Model params: {sum(p.numel() for p in _m.parameters()):,}')
del _m

## Cell 5 — FL Training Engine (FedAvg) + Sliding-Window Gradient Store

In [ ]:
def get_state_vec(model: nn.Module) -> torch.Tensor:
    return torch.cat([p.detach().flatten() for p in model.parameters()]).cpu()

def set_state_vec(model: nn.Module, vec: torch.Tensor):
    off = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(vec[off:off+n].view_as(p).to(p.device))
        off += n

def local_train(model: nn.Module, X: torch.Tensor, y: torch.Tensor, epochs: int, lr: float, bs: int, poison_label_flip: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
    if poison_label_flip:
        # Label-flip attack: flip 70% of labels uniformly at random within attack classes
        y = y.clone(); rng_local = np.random.RandomState(SEED + 1)
        flip_mask = rng_local.rand(len(y)) < 0.7
        y[flip_mask] = torch.tensor(rng_local.randint(0, n_classes, size=flip_mask.sum()), dtype=torch.long)
    pre = get_state_vec(model)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ds = TensorDataset(X, y); loader = DataLoader(ds, batch_size=bs, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); out = model(xb); loss = focal_loss(out, yb); loss.backward(); opt.step()
    post = get_state_vec(model)
    delta = post - pre
    return post, delta

def fed_avg(deltas: List[torch.Tensor], weights: List[float]) -> torch.Tensor:
    weights = np.array(weights, dtype=np.float64); weights = weights / weights.sum()
    out = torch.zeros_like(deltas[0])
    for d, w in zip(deltas, weights):
        out += d * float(w)
    return out

class GradientWindow:
    """Per-client sliding window of the last W gradient deltas — the SlideFU substrate."""
    def __init__(self, n_clients: int, window: int = CFG.window_W):
        self.W = window; self.buf: Dict[int, List[torch.Tensor]] = {i: [] for i in range(n_clients)}
    def push(self, client_id: int, delta: torch.Tensor):
        b = self.buf[client_id]; b.append(delta.detach().clone())
        if len(b) > self.W: b.pop(0)
    def history(self, client_id: int) -> List[torch.Tensor]:
        return self.buf[client_id]

def run_fl(global_model: nn.Module, n_rounds: int, poisoning_clients: List[int] = (),
           record_window: GradientWindow = None, log_every: int = 5) -> Dict:
    history = {'round': [], 'acc': [], 'f1': [], 'fpr': []}
    for r in range(n_rounds):
        deltas, weights = [], []
        gvec = get_state_vec(global_model)
        for cid in range(CFG.n_clients):
            local = copy.deepcopy(global_model)
            Xc, yc = client_data[cid]
            poison = (cid in poisoning_clients) and (r < CFG.poison_rounds)
            _, delta = local_train(local, Xc, yc, CFG.local_epochs, CFG.lr, CFG.batch_size, poison_label_flip=poison)
            deltas.append(delta); weights.append(len(yc))
            if record_window is not None:
                record_window.push(cid, delta)
        agg = fed_avg(deltas, weights)
        new = gvec + agg
        set_state_vec(global_model, new)
        if (r + 1) % log_every == 0 or r == n_rounds - 1:
            acc, f1, fpr = evaluate(global_model, X_te_t, y_te_t)
            history['round'].append(r+1); history['acc'].append(acc); history['f1'].append(f1); history['fpr'].append(fpr)
            print(f'  round {r+1:3d}/{n_rounds}  acc={acc:.2f}%  F1={f1:.2f}  FPR={fpr:.2f}%')
    return history

print('FL engine ready.')

## Cell 6 — Scenario A: FL Baseline (clean training)

In [ ]:
RESULTS = {}
print('=== Scenario A — FL Baseline ===')
model_A = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
win_A   = GradientWindow(CFG.n_clients)
t0 = time.time()
hist_A  = run_fl(model_A, CFG.n_rounds, poisoning_clients=[], record_window=win_A)
T_A = time.time() - t0
acc_A, f1_A, fpr_A = evaluate(model_A, X_te_t, y_te_t)
RESULTS['A'] = {'acc': acc_A, 'f1': f1_A, 'fpr': fpr_A, 'wallclock_s': T_A}
print(f'Scenario A final: acc={acc_A:.2f}%  F1={f1_A:.2f}  FPR={fpr_A:.2f}%  time={T_A:.1f}s')
# preserve a clean snapshot for reference
STATE_CLEAN = get_state_vec(model_A).clone()

## Cell 7 — Scenario B: FL with Poisoning (α = 10% label-flip)

In [ ]:
print('=== Scenario B — FL with Poisoning ===')
n_poison = max(1, int(round(CFG.poison_alpha * CFG.n_clients)))
poison_ids = sorted(np.random.RandomState(SEED).choice(CFG.n_clients, size=n_poison, replace=False).tolist())
print(f'Poisoning clients: {poison_ids}  (α={CFG.poison_alpha:.0%})')

model_B = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
win_B   = GradientWindow(CFG.n_clients)
t0 = time.time()
hist_B  = run_fl(model_B, CFG.n_rounds, poisoning_clients=poison_ids, record_window=win_B)
T_B = time.time() - t0
acc_B, f1_B, fpr_B = evaluate(model_B, X_te_t, y_te_t)
RESULTS['B'] = {'acc': acc_B, 'f1': f1_B, 'fpr': fpr_B, 'wallclock_s': T_B, 'poison_ids': poison_ids}
print(f'Scenario B final: acc={acc_B:.2f}%  F1={f1_B:.2f}  FPR={fpr_B:.2f}%  time={T_B:.1f}s')
STATE_POISONED = get_state_vec(model_B).clone()

## Cell 8 — Scenario C: FU-IDS Recovery via SlideFU calibration

Take the poisoned model from Scenario B, retrieve the gradient histories of the poisoning clients from the sliding window, and subtract their weighted contribution from the global state.

In [ ]:
def slidefu_calibrate(state_vec: torch.Tensor, window: GradientWindow,
                      target_clients: List[int], all_client_sizes: List[int]) -> torch.Tensor:
    """Reverse the contribution of `target_clients` by subtracting their weighted history.
    O(W) per target instead of O(T) full retrain."""
    total_size = sum(all_client_sizes)
    correction = torch.zeros_like(state_vec)
    for cid in target_clients:
        w_c = all_client_sizes[cid] / total_size
        for delta in window.history(cid):
            correction = correction + w_c * delta
    return state_vec - correction

print('=== Scenario C — FU-IDS Recovery (SlideFU) ===')
client_sizes = [len(yc) for _, yc in client_data]
t0 = time.time()
calibrated = slidefu_calibrate(STATE_POISONED, win_B, poison_ids, client_sizes)
model_C = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
set_state_vec(model_C, calibrated)
# brief fine-tune on remaining clients (1 round) — SlideFU's small recovery step
deltas, weights = [], []
for cid in range(CFG.n_clients):
    if cid in poison_ids:
        continue
    local = copy.deepcopy(model_C)
    Xc, yc = client_data[cid]
    _, d = local_train(local, Xc, yc, 1, CFG.lr, CFG.batch_size, poison_label_flip=False)
    deltas.append(d); weights.append(len(yc))
agg = fed_avg(deltas, weights)
set_state_vec(model_C, get_state_vec(model_C) + agg)
T_C = time.time() - t0
acc_C, f1_C, fpr_C = evaluate(model_C, X_te_t, y_te_t)
RESULTS['C'] = {'acc': acc_C, 'f1': f1_C, 'fpr': fpr_C, 'wallclock_s': T_C,
                'recovery_gap_pct': acc_C - acc_A,
                'cost_vs_retrain_pct': T_C / max(T_A, 1e-6) * 100}
print(f'Scenario C final: acc={acc_C:.2f}%  F1={f1_C:.2f}  FPR={fpr_C:.2f}%  time={T_C:.1f}s')
print(f'  Recovery gap (vs. Scenario A): {acc_C - acc_A:+.2f}%')
print(f'  Unlearning cost vs. full retrain: {T_C / max(T_A, 1e-6) * 100:.1f}% of retrain time')

## Cell 9 — Scenario D: BadUnlearn vs UnlearnGuard (PCA filter)

BadUnlearn: malicious clients craft updates *during the unlearning round itself* to inflate their own contribution and prevent recovery.
UnlearnGuard: PCA-projects calibration vectors and rejects out-of-distribution directions (cosine-similarity gating, τ = 0.65).

In [ ]:
def badunlearn_craft(window: GradientWindow, target_cids: List[int], inflation: float = 5.0) -> GradientWindow:
    """Replace target clients' window history with strongly inflated, gradient-aligned updates,
    so SlideFU subtraction overshoots and damages the model."""
    forged = copy.deepcopy(window)
    for cid in target_cids:
        for i, d in enumerate(forged.buf[cid]):
            forged.buf[cid][i] = d * inflation + torch.randn_like(d) * 0.05
    return forged

def unlearnguard_filter(window: GradientWindow, target_cids: List[int],
                        pca_components: int = CFG.pca_components, tau: float = CFG.trust_tau) -> GradientWindow:
    """PCA-project recent calibration vectors, gate by cosine similarity vs. honest centroid."""
    filtered = copy.deepcopy(window)
    # Build the 'honest direction' from non-target clients' last delta
    honest_vecs = []
    for cid, hist in window.buf.items():
        if cid not in target_cids and len(hist) > 0:
            honest_vecs.append(hist[-1].numpy())
    if not honest_vecs: return filtered
    H = np.stack(honest_vecs, axis=0)
    pca = PCA(n_components=min(pca_components, max(1, H.shape[0]-1)))
    Hp = pca.fit_transform(H)
    centroid = Hp.mean(axis=0)
    centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-9)
    # Filter target clients' history
    for cid in target_cids:
        new_hist = []
        for d in filtered.buf[cid]:
            v = pca.transform(d.numpy()[None, :])[0]
            v_norm = v / (np.linalg.norm(v) + 1e-9)
            cos = float(np.dot(v_norm, centroid_norm))
            if cos >= tau:
                new_hist.append(d)
            else:
                # rejected — substitute zero (i.e., this round's contribution is treated as ‘no info’)
                new_hist.append(torch.zeros_like(d))
        filtered.buf[cid] = new_hist
    return filtered

print('=== Scenario D — Adversarial Unlearning (BadUnlearn) ===')
# attempt 1: SlideFU with BadUnlearn-crafted history, NO defense
win_bad = badunlearn_craft(win_B, poison_ids, inflation=5.0)
calib_nodef = slidefu_calibrate(STATE_POISONED, win_bad, poison_ids, client_sizes)
model_D1 = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
set_state_vec(model_D1, calib_nodef)
acc_D1, f1_D1, fpr_D1 = evaluate(model_D1, X_te_t, y_te_t)
asr_nodef = max(0.0, (acc_A - acc_D1) / max(acc_A - acc_B, 1e-6)) * 100  # fraction of recovery prevented
print(f'No defense:  acc={acc_D1:.2f}%  ASR={asr_nodef:.1f}%')

# attempt 2: SlideFU with BadUnlearn + UnlearnGuard filter
win_filt = unlearnguard_filter(win_bad, poison_ids)
calib_def = slidefu_calibrate(STATE_POISONED, win_filt, poison_ids, client_sizes)
model_D2 = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
set_state_vec(model_D2, calib_def)
# brief fine-tune on remaining clients
deltas, weights = [], []
for cid in range(CFG.n_clients):
    if cid in poison_ids: continue
    local = copy.deepcopy(model_D2); Xc, yc = client_data[cid]
    _, d = local_train(local, Xc, yc, 1, CFG.lr, CFG.batch_size); deltas.append(d); weights.append(len(yc))
set_state_vec(model_D2, get_state_vec(model_D2) + fed_avg(deltas, weights))
acc_D2, f1_D2, fpr_D2 = evaluate(model_D2, X_te_t, y_te_t)
asr_def = max(0.0, (acc_A - acc_D2) / max(acc_A - acc_B, 1e-6)) * 100
print(f'UnlearnGuard: acc={acc_D2:.2f}%  ASR={asr_def:.1f}%')
RESULTS['D'] = {'no_defense': {'acc': acc_D1, 'asr': asr_nodef},
                'unlearnguard': {'acc': acc_D2, 'f1': f1_D2, 'fpr': fpr_D2, 'asr': asr_def}}

## Cell 10 — Scenario E: Privacy Certification ((ε, δ)-DP + MIA)

After unlearning, inject calibrated Gaussian noise to the global state and verify post-unlearning membership-inference resistance.

In [ ]:
def gaussian_dp_noise(state: torch.Tensor, epsilon: float, delta: float, sensitivity: float = 1.0) -> Tuple[torch.Tensor, float]:
    """Apply Gaussian mechanism noise scaled to (ε, δ). Returns (noisy_state, sigma)."""
    sigma = sensitivity * math.sqrt(2.0 * math.log(1.25 / delta)) / max(epsilon, 1e-3)
    return state + torch.randn_like(state) * sigma, sigma

def shadow_mia_auc(model: nn.Module, X_member: torch.Tensor, y_member: torch.Tensor,
                   X_nonmember: torch.Tensor, y_nonmember: torch.Tensor) -> float:
    """Cheap MIA proxy: confidence on member vs non-member predictions.
    AUC near 0.5 = no leakage; > 0.5 = leakage."""
    model.eval()
    def _conf(X, y):
        with torch.no_grad():
            logits = model(X.to(DEVICE)); probs = F.softmax(logits, dim=1)
            conf = probs[range(len(y)), y].cpu().numpy()
        return conf
    cm = _conf(X_member, y_member); cn = _conf(X_nonmember, y_nonmember)
    scores = np.concatenate([cm, cn])
    labels = np.concatenate([np.ones_like(cm), np.zeros_like(cn)])
    return float(roc_auc_score(labels, scores)) * 100

print('=== Scenario E — Privacy Certification ===')
# Apply DP noise to the unlearned model from Scenario C
noisy, sigma = gaussian_dp_noise(get_state_vec(model_C), CFG.dp_epsilon, CFG.dp_delta, sensitivity=1.0)
model_E = BiLSTMAttnIDS(n_features, n_classes).to(DEVICE)
set_state_vec(model_E, noisy)
acc_E, f1_E, fpr_E = evaluate(model_E, X_te_t, y_te_t)

# MIA setup: a poisoning client's data = 'members'; a slice of held-out test = 'non-members'
if poison_ids:
    Xm, ym = client_data[poison_ids[0]]
    Xm = Xm[: min(2000, len(Xm))]; ym = ym[: len(Xm)]
    # non-members: take from test set, same size
    Xn = X_te_t[: len(Xm)]; yn = y_te_t[: len(Xm)]
    auc_pre  = shadow_mia_auc(model_C, Xm, ym, Xn, yn)
    auc_post = shadow_mia_auc(model_E, Xm, ym, Xn, yn)
else:
    auc_pre = auc_post = 50.0

RESULTS['E'] = {'acc_after_dp': acc_E, 'sigma': sigma, 'epsilon': CFG.dp_epsilon, 'delta': CFG.dp_delta,
                'mia_auc_pre_dp': auc_pre, 'mia_auc_post_dp': auc_post}
print(f'Post-DP acc: {acc_E:.2f}%  |  σ={sigma:.4f}  |  MIA AUC pre={auc_pre:.1f}%  post={auc_post:.1f}%  (50% = random)')

## Cell 11 — Aggregate, format, and save final results

In [ ]:
summary = {
    'meta': {
        'data_source': DATA_SOURCE,
        'n_samples': len(df),
        'n_features': n_features,
        'n_classes': n_classes,
        'n_clients': CFG.n_clients,
        'n_rounds': CFG.n_rounds,
        'device': str(DEVICE),
        'full_experiment': FULL_EXPERIMENT,
        'seed': SEED,
    },
    'scenarios': RESULTS,
}
with open('FU_IDS_results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)

print('========== FU-IDS Experiment Summary ==========')
print(f'Data source: {DATA_SOURCE}  |  Device: {DEVICE}  |  Mode: {"FULL" if FULL_EXPERIMENT else "prototype"}')
print('-----------------------------------------------')
print(f'Scenario A (FL Baseline)        : acc={RESULTS["A"]["acc"]:.2f}%  F1={RESULTS["A"]["f1"]:.2f}  FPR={RESULTS["A"]["fpr"]:.2f}%  time={RESULTS["A"]["wallclock_s"]:.0f}s')
print(f'Scenario B (Poisoning, α=10%)   : acc={RESULTS["B"]["acc"]:.2f}%  F1={RESULTS["B"]["f1"]:.2f}  FPR={RESULTS["B"]["fpr"]:.2f}%  time={RESULTS["B"]["wallclock_s"]:.0f}s')
print(f'Scenario C (FU-IDS Recovery)    : acc={RESULTS["C"]["acc"]:.2f}%  F1={RESULTS["C"]["f1"]:.2f}  FPR={RESULTS["C"]["fpr"]:.2f}%  time={RESULTS["C"]["wallclock_s"]:.0f}s')
print(f'    Recovery gap vs Scenario A: {RESULTS["C"]["recovery_gap_pct"]:+.2f}%')
print(f'    Cost vs full retrain:        {RESULTS["C"]["cost_vs_retrain_pct"]:.1f}% of retrain time')
print(f'Scenario D (BadUnlearn) no def  : acc={RESULTS["D"]["no_defense"]["acc"]:.2f}%  ASR={RESULTS["D"]["no_defense"]["asr"]:.1f}%')
print(f'Scenario D + UnlearnGuard       : acc={RESULTS["D"]["unlearnguard"]["acc"]:.2f}%  ASR={RESULTS["D"]["unlearnguard"]["asr"]:.1f}%')
print(f'Scenario E (DP + MIA)           : acc={RESULTS["E"]["acc_after_dp"]:.2f}%  ε={RESULTS["E"]["epsilon"]}  δ={RESULTS["E"]["delta"]}  σ={RESULTS["E"]["sigma"]:.4f}')
print(f'    MIA AUC: pre-DP={RESULTS["E"]["mia_auc_pre_dp"]:.1f}%  post-DP={RESULTS["E"]["mia_auc_post_dp"]:.1f}%  (50% = ideal)')
print('-----------------------------------------------')
print('Saved JSON: FU_IDS_results.json')
print('Paste the summary above into the chat — we will integrate it into Section 7 of the paper.')